# Python → TypeScript Code Generator

A small tool for **converting Python to TypeScript**, **adding JSDoc and comments**, and **generating unit tests** using LLMs. Uses OpenAI (GPT-4o-mini) or Llama 3.2 (Ollama) behind a simple Gradio UI.

---

## Overview

This notebook lets you paste Python or TypeScript code, pick an action, choose a provider (OpenAI or Ollama), and get back code-only output—no long explanations, so you can copy and use the result directly.

---

## Key Features

* **Code-only output**
  * No markdown or prose; responses are unwrapped from code fences when present
  * Ready to paste into your project

* **Three actions**
  * **Convert to TypeScript** – Reimplement Python in typed TypeScript with the same behavior
  * **Add JSDoc & comments** – Add JSDoc and brief comments to existing TS/JS
  * **Write unit tests** – Generate Jest/Vitest tests in TypeScript

* **Dual provider**
  * OpenAI (GPT-4o-mini) when `OPENAI_API_KEY` is set in `.env`
  * Ollama (Llama 3.2) for local use—run `ollama run llama3.2` first

* **Simple UI**
  * Gradio: one code box, action dropdown, provider radio, and Generate button

---

## Requirements

* **Python:** `python-dotenv`, `openai`, `gradio`
* **OpenAI (optional):** `OPENAI_API_KEY` in `.env`
* **Ollama (optional):** Ollama running locally with `llama3.2` pulled

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
load_dotenv(override=True)
OPENAI_KEY = os.getenv("OPENAI_API_KEY")
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

openai = OpenAI() if OPENAI_KEY and OPENAI_KEY.startswith("sk-") and len(OPENAI_KEY) > 10 else None
MODELS = {"OpenAI": ("gpt-4o-mini", openai), "Ollama": ("llama3.2", ollama)}

print("OpenAI ready." if openai else "Set OPENAI_API_KEY in .env for GPT.")
print("Ollama ready (run: ollama run llama3.2).")

OpenAI ready.
Ollama ready (run: ollama run llama3.2).


## Code generator actions

Three actions: **Convert to TypeScript**, **Add JSDoc & comments**, and **Write unit tests** (Jest/Vitest).

In [3]:
ACTION_TO_PROMPT = {
    "Convert to TypeScript": (
        "You reimplement Python in TypeScript. Reply with only TypeScript: typed, no 'any', same behavior.",
        "Convert this Python to TypeScript. Output only the TypeScript code.",
    ),
    "Add JSDoc & comments": (
        "You add JSDoc and brief comments to TS/JS. Reply with only the updated code.",
        "Add JSDoc and comments. Output only the code.",
    ),
    "Write unit tests": (
        "You write Jest/Vitest tests in TypeScript. Reply with only the test file.",
        "Write unit tests. Output only the test code.",
    ),
}
ACTIONS = list(ACTION_TO_PROMPT)


def unwrap_fences(text: str) -> str:
    """Drop markdown code fence lines (first line if ```, last line if ```)."""
    text = (text or "").strip()
    if not text.startswith("```"):
        return text
    lines = text.splitlines()
    lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines)


def generate_code(code: str, action: str, provider: str) -> str:
    if not (code and code.strip()):
        return "Paste code first (Python for conversion, TypeScript for the rest)."
    if provider not in MODELS:
        return "Pick OpenAI or Ollama."
    model_id, client = MODELS[provider]
    if client is None:
        return "OpenAI not configured (OPENAI_API_KEY in .env)."
    system, instruction = ACTION_TO_PROMPT.get(action, list(ACTION_TO_PROMPT.values())[0])
    body = f"{instruction}\n\n{code.strip()}"
    messages = [{"role": "system", "content": system}, {"role": "user", "content": body}]
    try:
        r = client.chat.completions.create(model=model_id, messages=messages, temperature=0.3)
        out = (r.choices[0].message.content or "").strip()
        return unwrap_fences(out)
    except Exception as e:
        return f"Error: {e}"

## Gradio UI

Paste Python code (for conversion) or TypeScript code (for docstrings/tests), pick an **action**, choose a **model**, and generate.

In [4]:
with gr.Blocks(theme=gr.themes.Soft(), title="Python → TypeScript (sammyloto)") as demo:
    gr.Markdown("# Python → TypeScript\nPaste code, choose action and provider, then Generate.")
    with gr.Row():
        action = gr.Dropdown(ACTIONS, value=ACTIONS[0], label="Action")
        provider = gr.Radio(list(MODELS), label="Provider", value="OpenAI")
    code_in = gr.Textbox(label="Code", placeholder="Python or TypeScript…", lines=12)
    out = gr.Textbox(label="Output", lines=14)
    gr.Button("Generate", variant="primary").click(generate_code, [code_in, action, provider], out)

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
